**Objective**
Train and evaluate baseline classifiers for each dataset independently to establish benchmark performance for downstream AI Trust Score computation.

Inputs
data/processed/

feature_driver_drowsiness.csv
feature_driver_distraction.csv
feature_fairface.csv
feature_celeba.csv

Outputs:
outputs/

models/
metrics/
predictions/

In [ ]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

In [ ]:
DATA = pd.read_csv(
    "../data/processed/feature_driver_drowsiness.csv"
)

In [ ]:
FEATURES = [

    "brightness",

    "contrast",

    "blur_score",

    "image_area",

    "size_per_pixel",

    "Left_eye_react",

    "Right_eye_react"

]

In [ ]:
encoder = LabelEncoder()

DATA["target"] = encoder.fit_transform(
    DATA["Label"]
)

X = DATA[FEATURES]

y = DATA["target"]

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    y,

    test_size=0.30,

    random_state=42,

    stratify=y

)

X_valid, X_test, y_valid, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    random_state=42,

    stratify=y_temp

)

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_grid = {

    "n_estimators":[100,200,300],

    "max_depth":[5,10,20,None],

    "min_samples_leaf":[1,2,4],

    "min_samples_split":[2,5,10],

    "class_weight":[None,"balanced"]

}

rf_search = RandomizedSearchCV(

    rf,

    rf_grid,

    cv=5,

    n_iter=15,

    scoring="f1_weighted",

    random_state=42,

    n_jobs=-1

)

rf_search.fit(
    X_train,
    y_train
)

best_rf = rf_search.best_estimator_

In [ ]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_grid = {

    "n_estimators":[100,200,300],

    "learning_rate":[0.01,0.05,0.1],

    "max_depth":[3,5,7],

    "subsample":[0.8,1],

    "colsample_bytree":[0.8,1]

}

xgb_search = RandomizedSearchCV(

    xgb,

    xgb_grid,

    cv=5,

    scoring="f1_weighted",

    n_iter=15,

    random_state=42,

    n_jobs=-1

)

xgb_search.fit(
    X_train,
    y_train
)

best_xgb = xgb_search.best_estimator_

In [ ]:
lgb = LGBMClassifier(random_state=42)

lgb_grid = {

    "n_estimators":[100,200,300],

    "learning_rate":[0.01,0.05,0.1],

    "num_leaves":[31,50,70],

    "max_depth":[-1,5,10],

    "subsample":[0.8,1]

}

lgb_search = RandomizedSearchCV(

    lgb,

    lgb_grid,

    cv=5,

    scoring="f1_weighted",

    n_iter=15,

    random_state=42,

    n_jobs=-1

)

lgb_search.fit(
    X_train,
    y_train
)

best_lgb = lgb_search.best_estimator_

In [ ]:
models = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb
}

results = []

for name, model in models.items():

    pred = model.predict(X_valid)

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(y_valid, pred),

        "Precision": precision_score(y_valid, pred, average="weighted"),

        "Recall": recall_score(y_valid, pred, average="weighted"),

        "F1": f1_score(y_valid, pred, average="weighted")

    })

comparison = pd.DataFrame(results)

comparison.sort_values(
    "F1",
    ascending=False,
    inplace=True
)

comparison

In [ ]:
winner = comparison.iloc[0]["Model"]

best_model = models[winner]

test_pred = best_model.predict(X_test)

print(classification_report(
    y_test,
    test_pred,
    target_names=encoder.classes_
))

In [ ]:
OUTPUT_MODEL = Path("../outputs/models")
OUTPUT_MODEL.mkdir(parents=True, exist_ok=True)

OUTPUT_METRICS = Path("../outputs/metrics")
OUTPUT_METRICS.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_model,
    OUTPUT_MODEL / "best_model.pkl"
)

comparison.to_csv(
    OUTPUT_METRICS / "model_comparison.csv",
    index=False
)